In [1]:
!python -m pip install lightning
!pip install torchvision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 51.3 MB/s eta 0:00:00


In [50]:
import torch
import torch.nn as nn
import lightning as L
from torchvision.datasets import MNIST
from torchvision import datasets
from torch.utils.data import DataLoader
from torchvision import transforms
import torch, torch.nn as nn, torch.utils.data as data, torchvision as tv, torch.nn.functional as F
import torchvision.models as m
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights


In [48]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [49]:
train = datasets.MNIST(
    "./data",
    train=True,
    download=True,
    transform=transform
)

val = datasets.MNIST(
    "./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train,
    batch_size=64,
    shuffle=True,
    num_workers=4
)

val_loader = DataLoader(
    val,
    batch_size=64,
    shuffle=False,
    num_workers=4
)

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 490kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.99MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.4MB/s]
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [68]:
class LitEfficientNet(L.LightningModule):

    def __init__(self , num_classes = 10):
        super().__init__()

        self.model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        in_features = self.model.classifier[1].in_features

        self.model.classifier[1] = nn.Linear(
            in_features,
            num_classes
        )

        self.criterion = nn.CrossEntropyLoss()

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
          x, y = batch

          logits = self(x)

          loss = self.loss_fn(logits, y)

          self.log(
              "train_loss",
              loss,
              prog_bar=True
          )

          return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch

        logits = self(x)

        loss = self.loss_fn(logits, y)

        self.log(
            "val_loss",
            loss,
            prog_bar=True
        )
    def predict_step(self, batch, batch_idx):
      x, y = batch

      logits = self(x)

      return logits


    def configure_optimizers(self):

        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=1e-4
        )

        return optimizer

In [69]:
model = LitEfficientNet(num_classes=10)
model.loss_fn = nn.CrossEntropyLoss()

In [70]:
Trainer = L.Trainer(accelerator='gpu' , devices=1 , min_epochs=1 , max_epochs=3 , precision=16)
Trainer.fit(model, train_loader , val_loader )

/usr/local/lib/python3.13/dist-packages/lightning/fabric/connector.py:571: `precision=16` is supported for historical reasons but its usage is discouraged. Please set your precision to 16-mixed instead!
INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) 

┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ EfficientNet     │  4.0 M │ train │     0 │
│ 1 │ criterion │ CrossEntropyLoss │      0 │ train │     0 │
│ 2 │ loss_fn   │ CrossEntropyLoss │      0 │ train │     0 │
└───┴───────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 4.0 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.0 M                                                                                                
Total estimated model params size (MB): 16.081                                                                     
Modules in train mode: 339                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO: `Trainer.fit` stopped: `max_epochs=3` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=3` reached.


In [71]:
Trainer.logged_metrics

{'train_loss': tensor(0.0823), 'val_loss': tensor(0.0178)}

In [72]:
Trainer.predict(model , val_loader)

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

[tensor([[-2.4746e+00, -7.0020e-01, -1.5117e+00, -2.7383e+00, -4.7559e-01,
          -2.8027e+00, -4.5312e+00,  1.1711e+01, -4.9375e+00, -6.8604e-01],
         [-1.5781e+00, -5.2881e-01,  1.0727e+01, -2.9922e+00, -4.5977e+00,
          -4.9688e+00, -5.8594e-01, -2.7754e+00, -2.3438e+00, -5.3086e+00],
         [-2.2051e+00,  1.1297e+01, -1.6943e+00, -3.5762e+00, -1.7773e+00,
          -2.5137e+00, -1.3164e+00, -9.9756e-01, -2.9902e+00, -3.8438e+00],
         [ 1.0539e+01, -3.3672e+00, -3.1191e+00, -3.0273e+00, -3.6504e+00,
          -1.8672e+00,  1.8115e-01, -2.3516e+00, -1.8691e+00, -2.8394e-01],
         [-2.1094e+00, -2.9121e+00, -3.7598e-01, -4.3438e+00,  9.7500e+00,
          -2.3691e+00, -6.0889e-01, -3.8867e+00, -2.4043e+00, -1.3330e+00],
         [-2.1426e+00,  1.1289e+01, -1.6768e+00, -3.6133e+00, -2.1113e+00,
          -2.7617e+00, -2.1543e+00, -1.6858e-01, -2.9141e+00, -2.8301e+00],
         [-3.4355e+00, -2.1484e+00, -1.9043e+00, -4.5859e+00,  9.9141e+00,
          -1.5928e+

In [75]:
Trainer.save_checkpoint("model.ckpt")

INFO: `weights_only` was not set, defaulting to `False`.
INFO:lightning.pytorch.trainer.connectors.checkpoint_connector:`weights_only` was not set, defaulting to `False`.
